In [ ]:
import os
import boto3
from pathlib import Path
from dotenv import load_dotenv

# Load env variables from the dev.aws.env file
env_path = Path.cwd().parent.parent.joinpath("env").joinpath("dev.aws.env")

if env_path.exists():
    load_dotenv(dotenv_path=env_path, override=True)
    print(f"✓ Env loaded successfully. App Name: {os.getenv('APP_NAME')}")
else:
    print("✗ dev.aws.env file not found!")

# Initialize API Gateway V2 client
apigateway_client = boto3.client(
    "apigatewayv2",
    endpoint_url=os.getenv("AWS_URL"),
    region_name=os.getenv("REGION"),
    aws_access_key_id=os.getenv("AWS_ACCESS_KEY_ID"),
    aws_secret_access_key=os.getenv("AWS_SECRET_ACCESS_KEY")
)
print("apigatewayv2 client initialized successfully!")

In [ ]:
# Create a directory for our ExpressJS project
import os
os.makedirs("express-app", exist_ok=True)
print("✓ Created 'express-app' directory.")

In [ ]:
# Create the HTTP API Gateway
try:
    response = apigateway_client.create_api(
        Name="ExpressJS-Integration-API",
        ProtocolType="HTTP",
        Description="API Gateway HTTP API integration with ExpressJS"
    )
    api_id = response['ApiId']
    api_endpoint = response['ApiEndpoint']
    print(f"✓ Successfully created API Gateway HTTP API!")
    print(f"API Name: {response['Name']}")
    print(f"API ID: {api_id}")
    print(f"API Endpoint (LocalStack): {api_endpoint}")
except Exception as e:
    print(f"✗ Error creating API Gateway: {e}")

In [ ]:
# Define integration URI 
# In local environments (like LocalStack running inside Docker), 
# 'host.docker.internal' is typically used to reference the host system where Express.js runs on port 3000.
# If running LocalStack natively or using host-networking, use 'http://localhost:3000'.
INTEGRATION_URI = "http://host.docker.internal:3000"

try:
    integration_response = apigateway_client.create_integration(
        ApiId=api_id,
        IntegrationType="HTTP_PROXY",
        IntegrationMethod="ANY",
        IntegrationUri=INTEGRATION_URI,
        PayloadFormatVersion="2.0",
        Description="Integration forwarding requests to local ExpressJS server"
    )
    integration_id = integration_response['IntegrationId']
    print(f"✓ Successfully created Integration!")
    print(f"Integration ID: {integration_id}")
    print(f"Target URI: {INTEGRATION_URI}")
except Exception as e:
    print(f"✗ Error creating Integration: {e}")

In [ ]:
# Create a route that forwards all traffic to our integration
try:
    route_response = apigateway_client.create_route(
        ApiId=api_id,
        RouteKey="ANY /{proxy+}",
        Target=f"integrations/{integration_id}"
    )
    route_id = route_response['RouteId']
    print(f"✓ Successfully created Route!")
    print(f"Route ID: {route_id}")
    print(f"Route Key: {route_response['RouteKey']}")
except Exception as e:
    print(f"✗ Error creating Route: {e}")

# Create $default stage with auto-deploy enabled
try:
    stage_response = apigateway_client.create_stage(
        ApiId=api_id,
        StageName="$default",
        AutoDeploy=True,
        Description="Default auto-deployed stage"
    )
    print(f"✓ Successfully created $default Stage!")
    print(f"Stage Name: {stage_response['StageName']}")
    
    # Format endpoint URLs for verification
    # Standard localstack format is http://localhost:4566/restapis/<api_id>/$default/_/path
    # or using host names http://<api_id>.execute-api.localhost.localstack.cloud:4566/$default/hello
    print(f"\nTest URL Pattern (LocalStack): http://localhost:4566/restapis/{api_id}/$default/_/hello")
except Exception as e:
    print(f"✗ Error creating Stage: {e}")

In [ ]:
# Test sending a request to the API Gateway endpoint
import requests

try:
    test_endpoint = f"http://localhost:4566/restapis/{api_id}/$default/_/hello"
    print(f"Sending GET request to: {test_endpoint}")
    
    response = requests.get(test_endpoint, timeout=5)
    print(f"Status Code: {response.status_code}")
    print("Response Body:")
    print(response.text)
except Exception as e:
    print(f"✗ Failed to connect to API Gateway/Express.js server: {e}")
    print("Make sure the ExpressJS server is running on port 3000!")

In [ ]:
# Cleanup resources
try:
    print(f"Deleting API Gateway with ID: {api_id}")
    apigateway_client.delete_api(ApiId=api_id)
    print("✓ Successfully deleted API Gateway!")
except Exception as e:
    print(f"✗ Error deleting API Gateway: {e}")